# Sesión 3 · De un índice vectorial a una base de datos vectorial

En la primera sesión construimos el espacio de búsqueda: cada producto quedó representado mediante un vector E5 de 384 dimensiones y aprendimos a interpretar qué significa que dos elementos aparezcan cerca. En la segunda añadimos un mecanismo de acceso: comparamos la búsqueda exacta con distintos índices ANN y medimos qué vecinos se perdían al reducir el trabajo necesario para consultar el catálogo.

Ese recorrido todavía no da lugar a una base de datos vectorial. FAISS puede encontrar los vectores más próximos, pero no sabe por sí solo qué documento corresponde a cada posición, cómo interpretar un campo como `brand`, qué usuarios pueden acceder a una colección, cuándo una escritura pasa a ser visible o cómo recuperar el servicio tras una caída.

En el momento en que añadimos esas responsabilidades alrededor del índice, dejamos de trabajar únicamente con un algoritmo de búsqueda y empezamos a diseñar una base de datos vectorial. La calidad del sistema ya no depende solo del recall o de la latencia del ANN, sino también de cómo se gestionan los IDs, los metadatos, la persistencia, la concurrencia, la seguridad y el ciclo de vida de los datos.

En esta sesión no intentaremos decidir cuál es “el mejor proveedor” en términos absolutos. Definiremos primero un **contrato observable** y lo ejecutaremos sobre cinco motores. La comparación no buscará premiar al que produzca la demo más vistosa, sino entender qué responsabilidades asume cada solución, cuáles siguen recayendo sobre la aplicación y qué requisito concreto podría justificar una elección frente a otra.

<a id="s03-00-contrato"></a>

<a id="s03-00-indice"></a>

## Índice de contenidos

1. [Contrato común](#s03-00-contrato)
2. [Base de datos vectorial](#s03-00-base-de-datos)
3. [Consultas de referencia](#s03-00-consultas)
4. [Modelo de datos](#s03-00-modelo)
5. [Escrituras y consistencia](#s03-00-escrituras)
6. [Particiones, shards y réplicas](#s03-00-particiones)
7. [Despliegue gestionado y local](#s03-00-despliegue)
8. [Seguridad y observabilidad](#s03-00-seguridad)
9. [Protocolo de los laboratorios](#s03-00-protocolo)
10. [Decisión](#s03-00-decision)

## 1. Contrato común

Para que la comparación entre motores sea interpretable, mantendremos constantes todas las piezas que no forman parte del experimento. Cambiar al mismo tiempo el modelo, los datos, la métrica y la base vectorial impediría saber qué decisión explica cualquier diferencia observada.

Trabajaremos, por tanto, con el mismo snapshot utilizado en la sesión 2: 50.000 productos del catálogo, 276 consultas y embeddings `float32` normalizados generados con `intfloat/multilingual-e5-small`. Los documentos se codificaron con el prefijo `passage:` y las consultas con `query:`, respetando el contrato de entrada del modelo.

Cada base recibirá esos vectores ya calculados mediante un enfoque **BYOV** (*bring your own vectors*). No delegaremos la inferencia en el proveedor porque en esta sesión queremos aislar la capa de almacenamiento y recuperación.

De este modo, si dos motores producen resultados distintos, podremos investigar diferencias en el índice, los filtros, la consistencia, la persistencia o el comportamiento del cliente. No estaremos comparando, de forma encubierta, encoders distintos ni pipelines de preprocesamiento incompatibles.

In [ ]:
from pathlib import Path
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vector_database_session import (
    ProviderRun,
    SearchHit,
    evaluate_run,
    exact_top_k,
    iter_record_batches,
    load_session_data,
    record_id_for_product,
    validate_resource_name,
    wait_until,
    write_provider_run,
)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env")
data = load_session_data(memory_map=True)
TELEVISOR_QUERY_ID = "semantic-101352"
TALADRO_QUERY_ID = "semantic-100455"
TOP_K = 10

In [ ]:
snapshot_summary = pd.Series({
    "productos": len(data.products),
    "consultas": len(data.queries),
    "dimensión": data.product_embeddings.shape[1],
    "dtype": str(data.product_embeddings.dtype),
    "norma mínima": float(np.linalg.norm(data.product_embeddings, axis=1).min()),
    "norma máxima": float(np.linalg.norm(data.product_embeddings, axis=1).max()),
    "modelo": data.embedding_metadata["model_id"],
})
display(snapshot_summary.to_frame("valor"))

La dimensión, la normalización y la métrica también forman parte del esquema, aunque no aparezcan como columnas de negocio. Una colección creada para vectores de 1.536 dimensiones rechazará directamente estos embeddings de 384. Ese error es evidente. Más peligroso es configurar la colección con una métrica distinta: la escritura puede completarse sin problemas y la consulta devolver resultados plausibles, pero ordenados según otra geometría.

Por eso no basta con comprobar que la petición “funciona”. El contrato debe validar que la colección espera exactamente la misma dimensión, la misma política de normalización y la misma función de similitud con las que se generaron los vectores. Un desajuste en cualquiera de esas tres piezas puede convertir una integración aparentemente correcta en un sistema semánticamente incompatible.

También necesitamos una identidad estable para cada producto. `vector_id` describe únicamente la posición de una fila dentro de este snapshot y no debería utilizarse como identificador global. Si cambia el orden de los datos o se reconstruye la muestra, esa posición puede apuntar a otro producto.

Generaremos en su lugar un `record_id` mediante UUIDv5 a partir de `product_id` y del namespace fijo `3b4da450-802c-5dd6-80df-9bb588e5a4ac`. Como UUIDv5 es determinista, la misma entrada produce siempre el mismo identificador, con independencia del motor o de cuántas veces se repita la ingesta.

Esta propiedad hace que `upsert` sea idempotente: volver a cargar un producto actualiza el mismo registro en lugar de crear un duplicado. También simplifica la comparación entre motores, porque todos devuelven la misma identidad y no es necesario reconciliar cinco sistemas de IDs distintos.

In [ ]:
sample_records = next(iter_record_batches(data, batch_size=3))
pd.DataFrame([
    {
        **record.flat_metadata(),
        "text": record.text,
        "vector_dimensions": len(record.embedding),
    }
    for record in sample_records
])

<a id="s03-00-base-de-datos"></a>

## 2. Base de datos vectorial

El paso de FAISS a una base de datos vectorial no debe entenderse como la sustitución de un índice simple por una solución que resuelve mágicamente todos los problemas. Lo que cambia, sobre todo, es el reparto de responsabilidades.

FAISS se concentra en una tarea muy concreta: recibir vectores y encontrar vecinos según una métrica. Todo lo demás —la asociación con documentos, los filtros, las mutaciones, la persistencia, la concurrencia o la seguridad— debe construirse alrededor. Una base de datos vectorial toma muchas de esas decisiones y las convierte en parte explícita de su modelo operativo.

| Necesidad | Un índice FAISS aislado | Una base de datos vectorial |
|---|---|---|
| Vecinos por similitud | Sí | Sí, normalmente mediante un índice ANN interno |
| Documentos y metadatos | Mapeo externo de la aplicación | Modelo de datos del motor |
| Filtros durante la búsqueda | Lógica adicional o índices propios | Lenguaje de filtros e índices escalares |
| Mutaciones | Reconstrucción o tratamiento específico | `upsert`, `update`, `delete` con semántica declarada |
| Persistencia | Serializar, versionar y cargar archivos | WAL, segmentos o snapshots, según el motor |
| Concurrencia y API | Las implementa la aplicación | Servicio con clientes, límites y semántica propia |
| Particionado y réplicas | Diseño externo | Configuración de colección o servicio |
| Seguridad y auditoría | Fuera de FAISS | Autenticación, autorización, cifrado y logs |
| Observabilidad | Instrumentación propia | Métricas y estado, con distinta profundidad |

La tabla no implica que una base de datos vectorial sea automáticamente más robusta que una solución construida sobre FAISS. Indica que esas responsabilidades siguen existiendo, pero ahora tienen un lugar explícito dentro del sistema. El motor define cómo se aplican los filtros, cuándo una escritura se considera visible, cómo se persisten los datos o qué garantías ofrece ante una caída.

La diferencia también afecta a la operación. Un servicio gestionado delega parte de esa complejidad en un proveedor, pero introduce dependencia de sus límites, costes y contratos. Un despliegue propio conserva más control, aunque devuelve a nuestro equipo la responsabilidad sobre capacidad, backups, actualizaciones, monitorización y guardias.

La pregunta no es, por tanto, si una base vectorial “hace más cosas” que FAISS. La pregunta útil es cuáles de esas responsabilidades queremos asumir dentro de la aplicación, cuáles queremos delegar en el motor y qué garantías necesitamos poder observar y verificar.

<a id="s03-00-planos"></a>

### 2.1. Plano de datos y plano de control

Dentro de una base de datos vectorial conviene separar dos familias de operaciones que suelen confundirse bajo una misma API.

El **plano de datos** atiende el tráfico habitual de la aplicación: escrituras, lecturas, búsquedas por similitud y filtros sobre metadatos. Es la parte del sistema que participa directamente en una petición de usuario y determina qué documentos se almacenan, cuáles pueden recuperarse y qué ranking se devuelve.

El **plano de control** gestiona la infraestructura sobre la que opera ese tráfico. Aquí aparecen tareas como crear colecciones, configurar índices, asignar shards y réplicas, administrar credenciales o eliminar recursos. Estas operaciones no forman parte de cada consulta, pero condicionan su disponibilidad, capacidad y comportamiento.

```mermaid
flowchart LR
    A["Aplicación"] --> B["Cliente / SDK"]
    B --> C["API de datos"]
    C --> D["Metadatos + payload"]
    C --> E["Índice vectorial"]
    F["Plano de control"] --> C
    F --> G["Shards, réplicas, seguridad"]
    H["Observabilidad"] -. mide .-> C
    H -. mide .-> E
```

La observabilidad atraviesa ambos planos. No basta con saber que una consulta devolvió un resultado incorrecto o terminó con error; necesitamos identificar en qué capa apareció el problema.

Decir que “ha fallado la base de datos” es demasiado impreciso. Puede que la petición nunca llegara al servicio, que la colección todavía no estuviera lista, que el filtro descartara el documento correcto o que el índice ANN no visitara la región donde se encontraba. También puede ocurrir que la búsqueda exacta devolviera ya un vecino poco relevante, en cuyo caso el problema no estaría en la aproximación ni en el motor, sino en los embeddings o en el propio contrato de relevancia.

Cada hipótesis exige una prueba distinta. Por eso separaremos cliente, API, filtros, almacenamiento, índice y ranking exacto: solo así podremos atribuir un fallo a la capa que realmente lo produjo.

<a id="s03-00-consultas"></a>

## 3. Consultas de referencia

Utilizaremos las mismas dos consultas narrativas en todos los motores, pero cada una pondrá a prueba una parte distinta del sistema.

La consulta sobre el televisor servirá para medir la fidelidad al espacio vectorial. Aquí la pregunta es sencilla: dado el mismo embedding de consulta y el mismo catálogo, ¿hasta qué punto reproduce cada motor el ranking que obtendríamos mediante una búsqueda exacta?

La consulta sobre el taladro añade una condición estructurada. Ya no basta con encontrar productos semánticamente próximos; también hay que respetar un filtro que modifica el conjunto de candidatos válidos. La pregunta pasa a ser sobre qué universo se calcula realmente el top-$k$ y en qué momento se aplica la restricción.

Antes de consultar ninguna base construiremos un oráculo exacto mediante producto escalar. Como todos los vectores están normalizados a norma uno, ese orden coincide con el que produciría la similitud coseno.

El oráculo no representa una verdad comercial ni garantiza que los resultados sean útiles para el usuario. Representa exactamente la geometría aprendida por el modelo. Si el vecino exacto ya es incorrecto, no podremos atribuir el error al índice aproximado ni a la base de datos.

In [ ]:
television_row = data.query_row(TELEVISOR_QUERY_ID)
television_vector = data.query_vector(TELEVISOR_QUERY_ID)

television_exact = exact_top_k(data, television_vector, k=TOP_K)

display(Markdown(f"**Consulta:** {television_row['query_text']}"))
pd.DataFrame([hit.as_dict() for hit in television_exact])[
    ["rank", "product_id", "title", "brand", "native_score"]
]

El primer vecino exacto resulta ser un mantel. No se trata de una errata en la tabla ni de un problema al mostrar los resultados: bajo el espacio definido por E5, su similitud coseno con la consulta es mayor que la de los televisores presentes en el catálogo.

Si Pinecone, Chroma o Qdrant devuelven ese mismo UUID, la base de datos habrá reproducido correctamente un vecino semánticamente malo. En ese caso, cambiar de motor no corregirá el problema, porque el error nace antes, en la representación aprendida por el encoder. Un índice aproximado podría devolver por casualidad otro producto más razonable, pero esa desviación no constituiría una mejora fiable: sería un fallo de fidelidad que, en este ejemplo concreto, habría ocultado otro fallo.

> **Atribución.** Si el proveedor devuelve el mantel y obtiene `recall@10 = 1` frente al oráculo, el error se explica por el modelo. Si el oráculo coloca un televisor en primera posición y el proveedor no lo recupera, habrá que investigar el índice, la métrica o su configuración. Si el UUID correcto regresa desde la base pero la aplicación muestra otro título, el fallo estará en la correspondencia entre IDs y metadatos.

In [ ]:
drill_row = data.query_row(TALADRO_QUERY_ID)
drill_vector = data.query_vector(TALADRO_QUERY_ID)

drill_global = exact_top_k(data, drill_vector, k=TOP_K)
drill_einhell = exact_top_k(data, drill_vector, k=TOP_K, brand="Einhell")

comparison = pd.DataFrame({
    "ranking_global": [hit.product_id for hit in drill_global],
    "marca_global": [hit.brand for hit in drill_global],
    "ranking_condicionado": [hit.product_id for hit in drill_einhell],
    "marca_condicionada": [hit.brand for hit in drill_einhell],
}, index=np.arange(1, TOP_K + 1))
display(Markdown(f"**Consulta:** {drill_row['query_text']}"))
comparison

El filtro `brand == "Einhell"` no se limita a revisar el top-10 global y conservar los productos de esa marca. La condición redefine primero el universo de búsqueda: entre todos los productos Einhell, queremos encontrar los diez más próximos a la consulta.

La diferencia es importante. Si aplicáramos un postfiltro únicamente sobre los diez primeros vecinos globales, podríamos terminar con seis resultados aunque existieran muchos otros productos válidos en posiciones posteriores. Esos candidatos nunca llegarían a evaluarse porque el corte se habría realizado antes de aplicar la condición.

En un motor real, varias implementaciones pueden ofrecer el mismo contrato funcional y, sin embargo, comportarse de forma muy distinta. El sistema puede combinar índices de payload o bitmaps con HNSW, cambiar a búsqueda plana cuando el subconjunto permitido es pequeño, ampliar progresivamente la exploración hasta completar el cupo o recurrir a un postfiltro interno.

Desde fuera, todas esas estrategias pueden devolver diez productos Einhell. Por dentro, cambian la latencia, el trabajo realizado y el recall condicionado al filtro. Por eso no bastará con comprobar que la marca de los resultados es correcta.

Crearemos los índices de payload recomendados por cada motor y compararemos los IDs recuperados con un oráculo exacto construido sobre el mismo subconjunto de productos Einhell. De este modo podremos medir si la base encuentra realmente los vecinos más próximos dentro del universo permitido, y no solo si devuelve candidatos que cumplen la condición.

<a id="s03-00-modelo"></a>

## 4. Modelo de datos

Todos los motores recibirán el mismo modelo lógico. Cada registro contendrá `record_id`, `product_id`, `vector_id`, `title`, `brand`, `color`, `locale`, `text` y el embedding asociado. Antes de enviar los datos fuera de la aplicación, los valores nulos se convertirán en cadenas vacías.

Esta normalización evita diferencias silenciosas entre clientes. Un SDK podría omitir una clave por completo, otro serializarla como `null` y otro intentar enviar un `NaN` no válido. Si permitimos que cada proveedor interprete los nulos a su manera, dos colecciones supuestamente equivalentes pueden terminar almacenando documentos distintos y comportándose de forma diferente ante el mismo filtro.

Mantener un contrato lógico común no significa imponer la misma representación física. Cada motor organiza vectores, documentos y metadatos según su propio modelo de datos, y el adaptador debe traducir el registro común a esa forma nativa.

- En Pinecone, el vector y los metadatos escalares se almacenan dentro de un namespace. Incluiremos también `text` en los metadatos para poder inspeccionar el documento recuperado sin depender de un almacén externo durante la práctica.

- Chroma separa explícitamente `ids`, `embeddings`, `documents` y metadatos planos. El mismo producto se distribuye, por tanto, entre varios argumentos de la operación de escritura, aunque siga representando una única entidad lógica.

- Weaviate almacena las propiedades con tipos declarados y recibe el vector calculado por el cliente. Milvus exige definir un esquema con un campo vectorial y varios campos escalares, y expresa los filtros mediante su propio lenguaje de expresiones.

- Qdrant representa cada registro como un punto formado por un vector y un payload JSON. En este caso guardaremos `text` y `metadata` en rutas separadas para que LangChain pueda conectarse después sin obligarnos a migrar o reescribir la colección.

Estas diferencias pertenecen al adaptador de cada proveedor. Su responsabilidad consiste en preservar el contrato lógico mientras aprovecha la estructura nativa del motor. Si intentáramos forzar un JSON idéntico en todos ellos, terminaríamos ignorando capacidades propias, complicando los filtros o almacenando los datos de una forma innecesariamente ineficiente.

La comparación debe exigir equivalencia en el significado y en los resultados observables, no identidad en la representación interna.

In [ ]:
example = sample_records[0]
physical_shapes = {
    "pinecone": {"id": example.record_id, "values": "<384 float32>", "metadata": {**example.flat_metadata(), "text": example.text}},
    "chroma": {"id": example.record_id, "embedding": "<384 float32>", "document": example.text, "metadata": example.flat_metadata()},
    "qdrant": {"id": example.record_id, "vector": "<384 float32>", "payload": example.qdrant_payload()},
}
print(json.dumps(physical_shapes, ensure_ascii=False, indent=2)[:5000])

<a id="s03-00-escrituras"></a>

## 5. Escrituras y consistencia

En muchos motores, `upsert` significa insertar un registro nuevo o reemplazar el existente cuando el ID ya estaba presente. Sin embargo, que la operación termine correctamente no implica necesariamente que una búsqueda ejecutada justo después vaya a observar el cambio de inmediato.

En un motor local y monoproceso, la escritura puede hacerse visible prácticamente en el mismo instante. En un sistema distribuido, en cambio, la confirmación puede llegar antes de que todas las rutas de lectura, réplicas o estructuras de búsqueda hayan incorporado la nueva versión. La escritura ha sido aceptada, pero su visibilidad todavía puede estar propagándose.

Para observar ese comportamiento utilizaremos una secuencia de prueba controlada (un _canary test_). Insertaremos primero un UUID reservado para la sesión y comprobaremos que puede recuperarse tanto por ID como mediante una consulta vectorial. Después modificaremos uno de sus metadatos y esperaremos hasta observar la nueva versión. Por último, eliminaremos el registro y mediremos cuánto tarda en dejar de aparecer.

`wait_until()` no corrige ni refuerza la consistencia del motor. Su función es convertir la visibilidad en una propiedad medible: repite una condición, registra cuántos intentos fueron necesarios y cuánto tiempo transcurrió hasta que el cambio pudo observarse.

En Pinecone, esta prueba reflejará una experiencia compatible con consistencia eventual. En los SDKs que permiten solicitar `wait=True`, distinguiremos cuidadosamente entre haber pedido al cliente que espere a completar una operación y haber demostrado una garantía global de lectura tras escritura. Lo primero forma parte de la API; lo segundo solo puede afirmarse después de medir el comportamiento observable del sistema.

In [ ]:
temporary_test_id = record_id_for_product("S03-TEMPORARY-TEST")
visibility_sequence = pd.DataFrame([
    (1, "upsert", "El servidor confirma la operación"),
    (2, "fetch", "El ID y el payload son visibles"),
    (3, "query", "El punto aparece por similitud o filtro"),
    (4, "update", "La nueva versión de metadata es observable"),
    (5, "delete", "El punto deja de aparecer"),
], columns=["paso", "operación", "evidencia"])
visibility_sequence

<a id="s03-00-particiones"></a>

## 6. Particiones, shards y réplicas

Estos tres conceptos suelen aparecer juntos, pero responden a preguntas diferentes.

Una **colección** o un **namespace** sirve para separar datos de forma lógica. Por ejemplo, podríamos guardar los productos de cada país en una colección distinta o utilizar un namespace diferente para cada cliente.

Un **shard** divide una colección en varias partes para repartir los datos y el trabajo entre distintas máquinas. Esto permite almacenar más información o procesar más escrituras, aunque una consulta puede tener que buscar en varios shards y combinar después los resultados.

Una **réplica** es una copia adicional de los mismos datos. Se utiliza principalmente para mantener el servicio disponible si una máquina falla o para repartir las consultas de lectura. El coste es que cada réplica ocupa espacio y las escrituras deben propagarse entre todas las copias.

Por tanto, separar datos, dividirlos y copiarlos son decisiones distintas. Crear varias colecciones no sustituye a los shards, y añadir réplicas no divide el catálogo.

El aislamiento entre usuarios también puede resolverse de distintas formas. Una colección por usuario ofrece una separación clara, pero puede terminar creando miles de recursos difíciles de administrar. Una colección compartida con un campo `tenant_id` utiliza mejor la infraestructura, aunque la seguridad no debería depender de que la aplicación recuerde añadir ese filtro en cada consulta.

En esta sesión utilizaremos una única colección o índice por motor y, en Pinecone, un namespace para aislar los datos. Con solo 50.000 productos no intentaremos reproducir artificialmente un despliegue distribuido. Nos limitaremos a identificar dónde se configurarían los shards y las réplicas, y qué problemas resolvería cada uno.

<a id="s03-00-despliegue"></a>

## 7. Despliegue gestionado y local

En esta sesión utilizaremos Pinecone Serverless como opción gestionada y ejecutaremos Chroma, Weaviate, Milvus y Qdrant localmente mediante Docker. La comparación permitirá observar dos formas distintas de operar una base de datos vectorial, pero no servirá para declarar qué motor es más rápido.

Una petición a Pinecone atraviesa Internet y se ejecuta sobre infraestructura del proveedor. Los motores locales corren en nuestro propio equipo, con otra CPU, otra memoria, otra red y una caché diferente. También cambian la concurrencia disponible y los límites de capacidad. Comparar directamente sus milisegundos mezclaría demasiadas variables.

La diferencia relevante está en quién asume cada responsabilidad operativa.

| Decisión | Gestionado | Self-hosted |
|---|---|---|
| Aprovisionamiento | Se solicita mediante la API y queda sujeto a las cuotas del proveedor | Debemos proporcionar CPU, RAM, disco, red y orquestación |
| Upgrades | El proveedor gestiona la infraestructura y parte de la compatibilidad | Debemos planificar, probar y poder revertir cada actualización |
| Backups | Dependen del servicio y de la política contratada | Debemos crear las copias, definir su retención y comprobar que pueden restaurarse |
| Seguridad física y de plataforma | Se reparte entre el proveedor y el cliente | Una parte mayor de la plataforma queda bajo nuestra responsabilidad |
| Escalado | Se configura mediante las opciones del servicio y se paga según consumo | Debemos dimensionar capacidad, shards, réplicas y balanceo |
| Portabilidad | Queda condicionada por la API y el formato del proveedor | Conservamos más control sobre imágenes y datos, pero también toda la carga operativa |

Un servicio gestionado reduce el trabajo necesario para mantener la infraestructura, aunque obliga a aceptar sus contratos, límites y costes. Un despliegue propio ofrece más control, pero convierte las actualizaciones, los backups, la capacidad y la recuperación ante fallos en responsabilidades del equipo.

También conviene no confundir **local** con **producción**. Chroma embebido, Milvus Lite o Qdrant en `:memory:` son formas muy útiles de aprender la API, ejecutar pruebas y desarrollar integraciones. Sin embargo, no reproducen problemas como cortes de red, datos persistidos entre procesos, coordinación entre nodos, elección de líder o recuperación después de una caída.

Por tanto, utilizaremos ambos modos para estudiar sus contratos y sus interfaces, no para construir una clasificación de rendimiento entre entornos que no son comparables.

<a id="s03-00-seguridad"></a>

## 8. Seguridad y observabilidad

La seguridad no empieza en el documento original y termina cuando este se convierte en un vector. Un embedding puede conservar información sobre el texto del que procede, y los metadatos asociados pueden incluir atributos sensibles. Por eso una base de datos vectorial necesita los mismos controles que cualquier otro sistema de datos: autenticación, autorización por colección o tenant, cifrado en tránsito y en reposo, rotación de credenciales, auditoría y una política clara de borrado.

Guardar una API key directamente en un notebook puede ser aceptable durante una práctica controlada, pero no constituye un patrón de aplicación. En un sistema real, las credenciales deben permanecer fuera del código, limitar su alcance y poder revocarse sin reconstruir el servicio.

La observabilidad también debe cubrir algo más que la latencia. Para operar el buscador necesitaremos medir disponibilidad, errores por tipo de operación, percentiles de latencia, tasa de ingesta, retraso de indexación, recuento visible de registros, uso de disco y memoria, tamaño de segmentos y coste de los filtros. A esas métricas operativas hay que añadir una evaluación estable de recall.

Un dashboard puede mostrar una p95 excelente mientras el sistema recupera vecinos incorrectos. Del mismo modo, un recall alto medido en un notebook no dice nada sobre errores, saturación o retrasos de escritura bajo carga. La calidad y la operación deben observarse juntas.

Los informes `ProviderRun` conservarán el contexto necesario para interpretar cada ejecución: versión, destino, número de registros, tiempos descriptivos, semántica del score, IDs devueltos y mutaciones realizadas. El campo `native_score` se almacenará tal como lo entrega cada motor.

No intentaremos normalizar artificialmente esos valores. Una distancia y una similitud pueden producir exactamente el mismo ranking, pero sus escalas y su sentido son distintos. Comparar sus magnitudes como si fueran equivalentes ocultaría precisamente una parte del contrato que queremos observar.

In [ ]:
reports_directory = PROJECT_ROOT / ".artifacts" / "provider_runs"
report_paths = sorted(reports_directory.glob("*.json"))
report_rows = []
for report_path in report_paths:
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    report_rows.append({
        "provider": payload["provider"],
        "target": payload["target"],
        "records": payload["record_count"],
        "score_kind": payload["score_kind"],
        "higher_is_better": payload["higher_is_better"],
        "recall@10": payload["recall_at_k"],
    })
display(pd.DataFrame(report_rows) if report_rows else Markdown(
    "*Todavía no hay informes. Aparecerán aquí después de ejecutar los laboratorios.*"
))

<a id="s03-00-protocolo"></a>

## 9. Protocolo de los laboratorios

Cada notebook seguirá la misma secuencia de trabajo: comprobará primero que el entorno y las credenciales son válidos, creará los recursos de forma segura, realizará una ingesta idempotente y verificará el número de registros visibles. Después ejecutará la consulta del televisor, la consulta global del taladro y su versión filtrada, completará una prueba CRUD controlada, medirá la visibilidad de las mutaciones y comparará el top-10 con el oráculo exacto antes de generar el informe final.

El código será distinto en cada caso porque también lo son las APIs, los modelos de datos y las opciones de configuración. Pinecone, Chroma, Weaviate, Milvus y Qdrant no expresan de la misma forma una colección, un filtro o una escritura. Lo que permanecerá constante será la evidencia que exigimos: mismos datos, mismas consultas, mismos IDs y mismas comprobaciones funcionales.

La comparación final no debería terminar en una clasificación genérica de proveedores. Debe formularse a partir de los requisitos concretos del sistema:

- ¿Necesitamos delegar la operación en un servicio gestionado o el equipo puede mantener contenedores, actualizaciones y backups?
- ¿Qué volumen de datos, tasa de escritura, latencia p95, recall y disponibilidad debe soportar el buscador?
- ¿Qué filtros aplicaremos, cuán selectivos serán y cómo se combinarán?
- ¿Necesitamos aislamiento por tenant, múltiples vectores por documento, búsqueda sparse o híbrida, o una etapa posterior de reranking?
- ¿Cuánto puede tardar en hacerse visible una actualización de inventario o un borrado legal?
- ¿Qué herramientas posee ya el equipo para monitorizar, diagnosticar y recuperar el servicio?

Sin responder a estas preguntas, elegir por popularidad o por comodidad de la primera demo no elimina las decisiones difíciles. Solo las aplaza hasta que el sistema ya está integrado y cambiar de arquitectura resulta mucho más costoso.

<a id="s03-00-decision"></a>

## 10. Decisión

Para una clase, Pinecone reduce el trabajo de aprovisionamiento y permite concentrarse en el contrato que queremos observar. En una ruta local y ligera, Qdrant o Chroma ofrecen una experiencia más directa. Weaviate hace especialmente visible el modelo de propiedades y filtros, mientras que Milvus acerca la práctica a una arquitectura y un esquema pensados para cargas distribuidas.

Esta descripción no pretende construir un podio. Cada motor pone el énfasis en responsabilidades distintas, y su conveniencia depende del problema que deba resolver el equipo.

Con un catálogo de solo 50.000 vectores, cualquiera de estas soluciones puede resultar innecesaria si un proceso único con FAISS cumple el SLA y la aplicación ya resuelve correctamente la persistencia, los metadatos y el ciclo de vida. Añadir una base de datos vectorial no mejora el sistema por el mero hecho de introducir más componentes.

La base se justifica cuando necesitamos las responsabilidades adicionales que asume: filtros integrados, mutaciones, concurrencia, persistencia, replicación, seguridad o una operación más estructurada. La decisión debe partir de esas necesidades, no de la voluntad de que el diagrama de arquitectura contenga más cajas.

En los siguientes notebooks comprobaremos si cada motor conserva el contrato definido durante esta sesión. Después incorporaremos LangChain como una interfaz común y observaremos dos cosas: dónde simplifica la composición del sistema y en qué puntos vuelven a aparecer las particularidades de cada proveedor.
